In [1]:
# Ensure Homebrew binaries are in PATH
import os
os.environ['PATH'] = '/opt/homebrew/bin:' + os.environ.get('PATH', '')

In [2]:
# Install dependencies if needed
# !pip install langchain langchain-experimental langchain-chroma unstructured pydantic
import os
from textbook_loading_text_only import (
    load_book,
    clean_and_categorize_elements,
    summarize_elements,
    store_in_chromadb,
)

In [3]:
pdf_file = '../../data/Cat_Owners_Home_Veterinary_Handbook_Trimed.pdf'
chroma_persist_dir = '../../chroma/TO_Cat_Owners_Home_Veterinary_Handbook_Trimed/'

# Make sure the data directory exists
assert os.path.exists('../../data'), "Error: '../../data' directory not found."
assert os.path.exists(pdf_file), f"Error: PDF file not found at {pdf_file}."

#Small sample testing (works)
# pdf_file = '../../data/MediumExample_Ears_17Pgs.pdf'
# chroma_persist_dir = '../../chroma/TO_MediumExample_Ears_17Pgs/'

# # Make sure the data directory exists
# assert os.path.exists('../../data'), "Error: '../../data' directory not found."
# assert os.path.exists(pdf_file), f"Error: PDF file not found at {pdf_file}."

In [ ]:
print("📝 Unstructuring textbooks, filtering junks, semantic chunking...")
raw_pdf_elements = load_book(pdf_file)
print("🎉 1.process_pdf_with_semantic_chunking complete.")

📝 Unstructuring textbooks, filtering junks, semantic chunking...


In [ ]:
# Clean and categorize (text and tables only)
texts, tables = clean_and_categorize_elements(raw_pdf_elements, min_meaningful_text_length=75)

In [ ]:
# Check how many texts and tables we have
print(f"Number of text elements: {len(texts)}")
print(f"Number of table elements: {len(tables)}")
print(f"Total elements to summarize: {len(texts) + len(tables)}")

In [ ]:
# Summarize text and tables
text_summaries, table_summaries = summarize_elements(texts, tables, raw_pdf_elements)

In [ ]:
# Store in ChromaDB
retriever = store_in_chromadb(
    text_summaries, texts, table_summaries, tables,
    persist_directory=chroma_persist_dir
)

In [ ]:
# System sound, when done
sound_file = "/System/Library/Sounds/Glass.aiff"
os.system(f"afplay '{sound_file}'")

# Inspecting Retrieved Docs

In [ ]:
query = "My cat has being scratching its ear too often. There are some dark greasy thing in it. It scratch its ear so often and so hard that I see wounds and blood in it. What should I do?"
results = retriever.retrieve_multi_modal(query, k=5)

In [ ]:
# Display retrieved text chunks
print('-'*40, "Retrieved Text Chunks (first 300 chars)", '-'*40)
for res in results:
    if res["modality"] == "text":
        doc_id = res["original_metadata"].get("doc_id")
        original_text = None
        if doc_id and hasattr(retriever, "text_docstore"):
            doc = retriever.text_docstore._collection.get(ids=[doc_id], include=["documents"])
            if doc and doc.get("documents") and doc["documents"][0]:
                original_text = doc["documents"][0]
        if not original_text:
            original_text = res["summary"]
        text_display = original_text[:300] + ("..." if len(original_text) > 300 else "")
        print(text_display)
        print('-'*20)
    elif res["modality"] == "table":
        doc_id = res["original_metadata"].get("doc_id")
        print(f"[TABLE] Summary: {res['summary'][:200]}...")
        print('-'*20)